# Running Synthesizer on a Real TNG Galaxy

This is where things get real. You are about to:

1. Download real particle data for a simulated galaxy from the IllustrisTNG project
2. Load it into Python and look at the particles directly
3. Run **Synthesizer** — the same software Sophie uses in her research — to model the galaxy's light
4. Produce a spectrum showing how bright the galaxy is at each wavelength
5. Make a genuine 2D image as it would appear through a telescope

---

## Before you start: two things to check

### 1. Your TNG API key
You need this to download data. Ask Sophie for this.

### 2. The SPS grid file
Make sure the file `maraston24-Te00_kroupa-0.1,100.hdf5` is in the `grids/` folder
inside this `galaxy_explorer` directory. 

---

In [17]:
# ============================================================
#  STEP 0 — SET YOUR API KEY HERE
# ============================================================

API_KEY = "YOUR_API_KEY_HERE"   # ← Replace this with your key!

# ============================================================
#  You do not need to change anything below this line.
# ============================================================

# Load our helper functions (they live in helpers.py in this folder)
import sys, os
sys.path.insert(0, ".")
from helpers import (
    download_tng_galaxy,
    load_tng_particles,
    build_synthesizer_galaxy,
    get_spectrum,
    make_image,
    plot_spectrum,
    plot_one_image,
    plot_rgb,
    plot_particles,
)

import numpy as np
import matplotlib.pyplot as plt

print("Ready!")
print()
if API_KEY == "693fc85106de8da5bf5a9eebf34c3fc5":
    print("WARNING: You have not set your API key yet.")
    print("Replace 'YOUR_API_KEY_HERE' with your actual key and re-run this cell.")
else:
    print(f"API key set: {API_KEY[:8]}...{API_KEY[-4:]}  (looks good!)")

Ready!

API key set: YOUR_API...HERE  (looks good!)


---

## Step 1 — Choose a Galaxy

Every galaxy in TNG50 has a unique **subhalo ID** — its 'address' in the simulation.

We have pre-selected a few interesting galaxies to start with. You can explore more in Notebook 5.

| ID | Description |
|---|---|
| `117257` | A large spiral galaxy — good for a first look |
| `468590` | A compact, actively star-forming galaxy |
| `385350` | A massive, older 'red and dead' galaxy |
| `474170` | A smaller irregular galaxy |

Start with `553837`.

In [10]:
# Pick a galaxy!
# Try changing this ID in Notebook 5.

GALAXY_ID = 553837

print(f"Selected galaxy: TNG50-1, subhalo {GALAXY_ID}, snapshot 99 (redshift z=0)")

Selected galaxy: TNG50-1, subhalo 553837, snapshot 99 (redshift z=0)


---

## Step 2 — Download the Particle Data

TNG50 stores data as **HDF5 files** (a format designed for large scientific datasets).
We will download a 'cutout' — just the star particles belonging to our chosen galaxy.

This should be a few MB and take under a minute.

In [16]:
# Download the galaxy's particle data
# (The file is saved locally so you won't need to re-download it)

data_file = download_tng_galaxy(GALAXY_ID, api_key=API_KEY)
print(f"\nData file: {data_file}")

RuntimeError: Download failed with status code 403.
Check that the subhalo ID exists in TNG50-1 snapshot 99.

---

## Step 3 — Load and Inspect the Particles

Each TNG galaxy is represented by thousands of **star particles**. Each particle represents a large clump of stars that all formed at the same time from the same gas cloud (a 'stellar population').

For each particle, TNG tells us:
- **Position** in 3D space (x, y, z)
- **Initial mass** when the particle first formed
- **Formation time** (as a 'scale factor' — a measure of when in cosmic history it formed)
- **Metallicity** — what fraction of the particle's mass is in elements heavier than hydrogen and helium

In [ ]:
# Load the particle data and convert units
# (helpers.py handles all the unit conversions — kpc, Msun, years)

particles = load_tng_particles(data_file)

print()
print("Properties of the star particles:")
print(f"  Age range:         {particles['ages_yr'].min()/1e9:.2f} – {particles['ages_yr'].max()/1e9:.2f} Gyr")
print(f"  Mass range:        {particles['masses_msun'].min():.2e} – {particles['masses_msun'].max():.2e} Msun")
print(f"  Metallicity range: {particles['metallicity'].min():.5f} – {particles['metallicity'].max():.4f}")
print(f"  Spatial extent:    ~{particles['coords_kpc'].max() - particles['coords_kpc'].min():.0f} kpc across")

In [ ]:
# Let's look at the particles directly — plot their positions!
# This is the galaxy as the simulation 'sees' it, before any light modelling.

plot_particles(particles, title=f"TNG50 Galaxy {GALAXY_ID} — star particle positions",
               color_by="age")

In [ ]:
# Now colour by metallicity to see a different physical property
plot_particles(particles, title=f"TNG50 Galaxy {GALAXY_ID} — coloured by metallicity",
               color_by="metallicity")

**Think about what you see:**
- Where are the youngest stars (small ages)? Where are the oldest?
- Is the metallicity (amount of heavy elements) different in different parts of the galaxy?
- Which direction would be 'face-on' and which would be 'edge-on' if you were looking at this as a photograph?

---

## Step 4 — Build the Synthesizer Galaxy Object

Now we hand the particle data to **Synthesizer**. It will:
1. Attach proper physical units to everything
2. Load the SPS grid (the lookup table of stellar spectra)
3. Create a `Galaxy` object that Synthesizer can work with

Make sure `grids/maraston24-Te00_kroupa-0.1,100.hdf5` is in the right place!

In [ ]:
# Build the Synthesizer Galaxy object
# This loads the SPS grid and prepares the data

galaxy, grid = build_synthesizer_galaxy(
    particles,
    grid_dir  = "../grids",
    grid_name = "maraston24-Te00_kroupa-0.1,100"
)

print()
print("The SPS grid contains spectra for every combination of:")
print(f"  Ages:          {len(grid.ages):>4} values  ({grid.ages.min():.2e} – {grid.ages.max():.2e} yr)")
print(f"  Metallicities: {len(grid.metallicities):>4} values  ({grid.metallicities.min():.4f} – {grid.metallicities.max():.4f})")
print(f"  Wavelengths:   {len(grid.lam):>4} values")

---

## Step 5 — Run Synthesizer: Compute the Spectrum

This is the core step. Synthesizer:
- Loops over every star particle
- Looks up its spectrum in the SPS grid (by interpolating between grid points using the particle's age and metallicity)
- Weights the spectrum by the particle's mass
- Sums all particle spectra together → the total galaxy **Spectral Energy Distribution (SED)**

On a laptop with thousands of particles, this takes a few seconds.

In [ ]:
print("Running Synthesizer — this may take 10-30 seconds on a laptop...")
print()

sed, spectrum_label = get_spectrum(galaxy, grid)

print()
print(f"Total luminosity: {sed.bolometric_luminosity:.2e}")

In [ ]:
# Plot the galaxy's spectrum!
plot_spectrum(sed, title=f"TNG50 Galaxy {GALAXY_ID} — Spectral Energy Distribution")

**Think about what the spectrum is telling you:**
- At which wavelengths is this galaxy brightest?
- Is it a young, star-forming galaxy or an old one? (Young → UV-bright; Old → infrared-bright)
- How does this compare with the toy spectra in Notebook 3?

---

## Step 6 — Get Photometry (Telescope Filter Fluxes)

A real telescope does not measure the full spectrum — it measures the total brightness through a **filter** that only lets through a particular range of wavelengths.

For example:
- A **LSST r-band** filter lets through roughly 5500–7000 Å (red optical light)
- A **GALEX FUV** filter lets through around 1300–1800 Å (far ultraviolet)

Synthesizer can fold the galaxy's spectrum through any filter and return the flux.

In [ ]:
# Get photometry in several filters
# Synthesizer downloads filter curves automatically from the Spanish Virtual Observatory

from synthesizer.instruments import FilterCollection
from unyt import angstrom

filter_codes = [
    "GALEX/GALEX.FUV",       # Far UV
    "GALEX/GALEX.NUV",       # Near UV
    "SLOAN/SLOAN.u",         # SDSS u-band (violet)
    "SLOAN/SLOAN.g",         # SDSS g-band (green)
    "SLOAN/SLOAN.r",         # SDSS r-band (red)
    "SLOAN/SLOAN.i",         # SDSS i-band (near-IR)
    "2MASS/2MASS.J",          # Near-infrared J
    "2MASS/2MASS.K",          # Near-infrared K
]

print("Loading filter curves and computing photometry...")

fc = FilterCollection(filter_codes=filter_codes)
fc.calc_pivot_lam()   # compute central wavelength for each filter

# Compute the photometry: how bright is this galaxy through each filter?
galaxy.stars.spectra[spectrum_label].get_photo_luminosities(fc)
photometry = galaxy.stars.spectra[spectrum_label].photo_luminosities

print()
print("Galaxy luminosity through each filter:")
for code in filter_codes:
    filt_name = code.split('/')[-1]
    lum = photometry[code]
    print(f"  {filt_name:<15}  {float(lum):.2e} erg/s/Hz")

In [ ]:
# Plot the filter fluxes on top of the spectrum
import numpy as np
import matplotlib.pyplot as plt

lam_A = sed.lam.to("Angstrom").value
lum   = sed.luminosity

fig, ax = plt.subplots(figsize=(12, 5))
ax.loglog(lam_A, lum, color='steelblue', lw=1.2, alpha=0.8, label='Full spectrum (SED)')

# Overlay the filter fluxes as coloured points
filter_colours = ['purple','violet','royalblue','forestgreen','tomato','orange','darkred','firebrick']
for code, colour in zip(filter_codes, filter_colours):
    filt_name  = code.split('/')[-1]
    pivot_lam  = float(fc[code].pivot_lam.to("Angstrom").value)
    phot_flux  = float(photometry[code])
    ax.scatter(pivot_lam, phot_flux, s=120, color=colour, zorder=5,
               edgecolors='white', linewidths=0.7, label=filt_name)

ax.set_xlabel('Wavelength (Å)', fontsize=12)
ax.set_ylabel(r'Luminosity  (erg s$^{-1}$ Hz$^{-1}$)', fontsize=12)
ax.set_title(f'Galaxy {GALAXY_ID} — spectrum + filter photometry', fontsize=13)
ax.axvspan(3800, 7000, alpha=0.08, color='gold', label='Visible light')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Each dot shows what a telescope with that filter would measure.")
print("The dots sample the smooth spectrum — this is how colour is measured in astronomy.")

---

## Step 7 — Make a Galaxy Image

Now the exciting part! We will project the star particles onto a 2D pixel grid to create an image — the kind you would see through a telescope.

Synthesizer places each particle in the pixel that corresponds to its x-y position, weighted by how much light it produces through a given filter.

The `fov_kpc` parameter controls how wide the image is in kiloparsecs.
The `pixel_kpc` parameter controls the size of each pixel.

In [ ]:
# Make a galaxy image in the r-band (red optical light)
# This is what the Sloan Digital Sky Survey (SDSS) sees

print("Making galaxy image...")

images = make_image(
    galaxy,
    spectrum_label,
    fov_kpc    = 60.0,    # image width: 60 kpc across
    pixel_kpc  = 0.6,     # pixel size: 0.6 kpc
    filter_codes = ["SLOAN/SLOAN.r", "SLOAN/SLOAN.g", "SLOAN/SLOAN.u"],
)

print(f"Image produced! Shape: {images['SLOAN/SLOAN.r'].arr.shape} pixels")
print(f"Each pixel represents {0.6:.1f} kpc = {0.6 * 3260:.0f} light years")

In [ ]:
# Display the r-band image
plot_one_image(
    images["SLOAN/SLOAN.r"].arr,
    title=f"Galaxy {GALAXY_ID} — SDSS r-band (red optical light)",
    cmap="magma"
)

In [ ]:
# Compare the same galaxy in three different filters
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='black')

band_info = [
    ("SLOAN/SLOAN.u", "magma",   "UV / u-band\n(~ 3550 Å — shows HOT young stars)"),
    ("SLOAN/SLOAN.g", "magma",   "Optical / g-band\n(~ 4770 Å — shows all stars)"),
    ("SLOAN/SLOAN.r", "magma",   "Red / r-band\n(~ 6230 Å — shows older stars)"),
]

for ax, (code, cmap, label) in zip(axes, band_info):
    arr = np.array(images[code].arr)
    arr_log = np.where(arr > 0, np.log10(arr + 1e-40), np.nan)
    vmin = np.nanpercentile(arr_log, 5)
    vmax = np.nanpercentile(arr_log, 99.5)
    ax.imshow(arr_log, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(label, color='white', fontsize=10, pad=6)
    ax.axis('off')
    ax.set_facecolor('black')

fig.patch.set_facecolor('black')
fig.suptitle(f"Galaxy {GALAXY_ID} — Same galaxy, different wavelengths",
             color='white', fontsize=13)
plt.tight_layout()
plt.show()

print("\nDo the three images look different? If so, why?")
print("Think about which types of stars glow brightest at each wavelength.")

In [ ]:
# Make a beautiful RGB colour image by combining all three filters!

plot_rgb(
    images,
    r_filter = "SLOAN/SLOAN.r",
    g_filter = "SLOAN/SLOAN.g",
    b_filter = "SLOAN/SLOAN.u",
    title    = f"Galaxy {GALAXY_ID} — Synthesizer RGB image (r / g / u)",
    stretch  = 0.008
)

---

## Step 8 — Understanding What You Just Did

Let's recap the journey:

```
  TNG50 simulation              Synthesizer                    Your image
  ────────────────              ───────────                    ──────────
  {particle positions}          Grid lookup: age + Z → spectrum
  {particle masses}      ───►   Sum all particle spectra       ───►  galaxy.png
  {particle ages}               Fold through SDSS filter
  {particle metallicity}        Project onto 2D pixel grid
```

This is exactly what Sophie does when she runs her pipeline on COSMA.
The difference is that her pipeline processes **millions of galaxies** on a supercomputer with 5,000 CPU cores, using a higher-resolution grid.

---

**Congratulations!** You have just run a genuine astrophysical forward model.

**→ Open `05_explore_discover.ipynb` to start experimenting!**